In [ ]:
!pip install datasets pandas scikit-learn sentence-transformers transformers torch

import json
import pandas as pd
from typing import List, Dict, Set, Tuple
import re
import os
from pathlib import Path
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

print("All dependencies installed and imported!")

In [ ]:
class BIOSKGCOVIDSubset:
    def __init__(self, load_from_hf: bool = True, cache_dir: str = "./data"):
        self.load_from_hf = load_from_hf
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(exist_ok=True)

        self.kg_data = None
        self.covid_subset = []

        # COVID-19 related keywords for filtering
        self.covid_keywords = [
            'covid', 'covid-19', 'covid19', 'sars-cov-2', 'coronavirus',
            'sars-cov', 'mers-cov', '2019-ncov', 'hcov', 'cov-2',
            'pandemic', 'epidemic', 'respiratory syndrome'
        ]

        self.covid_drugs = [
            'nirmatrelvir', 'ritonavir', 'remdesivir', 'molnupiravir', 'sotrovimab',
            'dexamethasone', 'hydrocortisone', 'prednisolone', 'tocilizumab', 'baricitinib',
            'azithromycin', 'doxycycline', 'colchicine', 'ivermectin', 'budesonide',
            'tixagevimab', 'cilgavimab', 'paxlovid', 'lagevrio', 'actemra',
            'olumab', 'sarilumab', 'evusheld', 'bamlanivimab', 'casirivimab', 'imdevimab'
        ]

        self.covid_medical_terms = [
            'acute respiratory distress syndrome', 'ards',
            'cytokine storm', 'hyperinflammation', 'cytokine release syndrome',
            'pneumonia', 'respiratory failure', 'hypoxia', 'oxygen therapy',
            'oxygen saturation', 'ventilator', 'mechanical ventilation', 'icu',
            'intensive care', 'hospitalization', 'severe acute respiratory',
            'long covid', 'post-acute sequelae', 'pasc',
            'anosmia', 'ageusia', 'dyspnea', 'shortness of breath',
            'ground glass opacity', 'pulmonary embolism', 'thrombosis'
        ]

    def load_kg_from_huggingface(self):
        """Load BIOS KG from Hugging Face datasets."""
        print("Loading BIOS KG from Hugging Face...")
        print("Dataset: THUMedInfo/BIOS_v3")

        try:
            dataset = load_dataset("THUMedInfo/BIOS_v3", cache_dir=str(self.cache_dir))

            print(f"Loaded BIOS dataset with splits: {list(dataset.keys())}")

            all_data = []
            for split in dataset.keys():
                split_data = dataset[split]
                print(f"  {split}: {len(split_data)} triples")
                all_data.extend(split_data)

            self.kg_data = all_data
            print(f"Total triples loaded: {len(self.kg_data)}")
            return True

        except Exception as e:
            print(f"Error loading from Hugging Face: {e}")
            print("\n Troubleshooting tips:")
            print("1. Check internet connection")
            print("2. Make sure 'datasets' library is installed: pip install datasets")
            print("3. Try with smaller subset first")
            return False

In [ ]:
    def load_kg_from_local(self, file_path: str):
        """Load BIOS KG from local file."""
        print(f" Loading BIOS KG from local file: {file_path}")

        if not os.path.exists(file_path):
            print(f" File not found: {file_path}")
            return False

        try:
            if file_path.endswith('.json'):
                with open(file_path, 'r', encoding='utf-8') as f:
                    self.kg_data = json.load(f)
            elif file_path.endswith('.csv'):
                self.kg_data = pd.read_csv(file_path).to_dict('records')
            else:
                print(" Unsupported file format. Use JSON or CSV.")
                return False

            print(f" Loaded {len(self.kg_data)} triples from local file")
            return True

        except Exception as e:
            print(f" Error loading local file: {e}")
            return False

    def load_kg(self):
        """Main method to load KG data"""
        if self.load_from_hf:
            return self.load_kg_from_huggingface()
        else:
            local_paths = [
                "./data/BIOS_KG.json",
                "./data/BIOS_KG.csv",
                "./data/bios_kg.json",
                "./BIOS_KG.json"
            ]
            for path in local_paths:
                if os.path.exists(path):
                    return self.load_kg_from_local(path)

            print(" No local BIOS KG file found.")
            return False

    def explore_dataset_structure(self):
        """Explore the structure of the loaded BIOS dataset."""
        if not self.kg_data:
            print(" No data loaded. Call load_kg() first.")
            return

        print("\n Exploring BIOS dataset structure...")

        # Check the first few items to understand structure
        sample_item = self.kg_data[0] if self.kg_data else {}
        print(f"Sample item keys: {list(sample_item.keys())}")
        print(f"Sample item: {sample_item}")

        # Count different relation types
        if hasattr(self.kg_data, 'features'):
            print(f"Dataset features: {self.kg_data.features}")

        # Check for common column names
        if isinstance(self.kg_data, list) and len(self.kg_data) > 0:
            first_item = self.kg_data[0]
            possible_relation_keys = ['relation', 'predicate', 'rel', 'edge']
            possible_head_keys = ['head', 'subject', 'source', 'entity1']
            possible_tail_keys = ['tail', 'object', 'target', 'entity2']

            print(f"Available keys in first item: {list(first_item.keys())}")

In [ ]:
    def is_covid_related(self, text: str) -> bool:
        """Check if text contains COVID-19 related keywords."""
        if not text or not isinstance(text, str):
            return False

        text_lower = text.lower()

        # Check for exact COVID-19 terms with word boundaries
        covid_patterns = [
            r'\bcovid[-\s]?19\b',
            r'\bsars[-\s]cov[-\s]2\b',
            r'\b2019[-\s]ncov\b',
            r'\bcoronavirus\b',
            r'\bcorona virus\b'
        ]

        for pattern in covid_patterns:
            if re.search(pattern, text_lower, re.IGNORECASE):
                return True

        # Check for COVID-19 drugs
        for drug in self.covid_drugs:
            if drug.lower() in text_lower:
                return True

        # Check for medical terms in COVID context
        for term in self.covid_medical_terms:
            if term.lower() in text_lower:
                return True

        # Check general COVID keywords
        for keyword in self.covid_keywords:
            if keyword.lower() in text_lower:
                return True

        return False

    def extract_covid_subset(self, save_path: str = None) -> List[Dict]:
        """Extract COVID-19 related triples from BIOS KG."""
        if self.kg_data is None:
            if not self.load_kg():
                print(" Failed to load KG data.")
                return []

        print(" Extracting COVID-19 related triples...")

        covid_triples = []
        processed_count = 0

        for item in self.kg_data:
            processed_count += 1
            if processed_count % 100000 == 0:
                print(f"  Processed {processed_count} triples...")

            triple = self._process_kg_item(item)
            if triple and self._is_covid_triple(triple):
                covid_triples.append(triple)

        self.covid_subset = covid_triples
        print(f" Extracted {len(covid_triples)} COVID-19 related triples from {processed_count} total triples")

        if save_path:
            self.save_subset(save_path)

        return covid_triples

In [ ]:
    def _process_kg_item(self, item) -> Dict:
        """Process a KG item regardless of its structure."""
        if isinstance(item, dict):
            # Try different key naming conventions
            key_combinations = [
                (['head', 'relation', 'tail']),
                (['subject', 'predicate', 'object']),
                (['source', 'relation', 'target']),
                (['entity1', 'relation', 'entity2']),
                (['from', 'rel', 'to']),
                (['h', 'r', 't']),
            ]

            for h_key, r_key, t_key in key_combinations:
                if h_key in item and r_key in item and t_key in item:
                    return {
                        'head': str(item[h_key]),
                        'relation': str(item[r_key]),
                        'tail': str(item[t_key]),
                        'original_item': item
                    }

        # If it's a simple tuple-like structure
        elif isinstance(item, (list, tuple)) and len(item) >= 3:
            return {
                'head': str(item[0]),
                'relation': str(item[1]),
                'tail': str(item[2])
            }

        return None

    def _is_covid_triple(self, triple: Dict) -> bool:
        """Check if a triple is COVID-19 related."""
        head = str(triple.get('head', ''))
        relation = str(triple.get('relation', ''))
        tail = str(triple.get('tail', ''))

        return (self.is_covid_related(head) or
                self.is_covid_related(relation) or
                self.is_covid_related(tail))

    def save_subset(self, save_path: str):
        """Save the COVID-19 subset to file."""
        if not self.covid_subset:
            print(" No COVID-19 subset to save. Run extract_covid_subset first.")
            return

        # Ensure directory exists
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)

        file_ext = save_path.split('.')[-1].lower()

        try:
            if file_ext == 'json':
                with open(save_path, 'w', encoding='utf-8') as f:
                    json.dump(self.covid_subset, f, indent=2, ensure_ascii=False)
            elif file_ext == 'csv':
                # Simplify for CSV export
                simplified_data = []
                for triple in self.covid_subset:
                    simple_triple = {k: v for k, v in triple.items() if k != 'original_item'}
                    simplified_data.append(simple_triple)

                df = pd.DataFrame(simplified_data)
                df.to_csv(save_path, index=False)
            else:
                # Default to JSON
                json_path = save_path + '.json'
                with open(json_path, 'w', encoding='utf-8') as f:
                    json.dump(self.covid_subset, f, indent=2, ensure_ascii=False)

            print(f" COVID-19 subset saved to: {save_path}")

        except Exception as e:
            print(f" Error saving subset: {e}")

In [ ]:
    def analyze_subset(self):
        """Analyze the extracted COVID-19 subset."""
        if not self.covid_subset:
            print(" No COVID-19 subset to analyze. Run extract_covid_subset first.")
            return

        print("\n" + "="*60)
        print(" COVID-19 Subset Analysis")
        print("="*60)
        print(f"Total COVID-19 triples: {len(self.covid_subset):,}")

        # Count unique entities and relations
        heads = set(triple['head'] for triple in self.covid_subset)
        tails = set(triple['tail'] for triple in self.covid_subset)
        relations = set(triple['relation'] for triple in self.covid_subset)

        print(f"Unique head entities: {len(heads):,}")
        print(f"Unique tail entities: {len(tails):,}")
        print(f"Unique relations: {len(relations):,}")

        # Most common relations
        relation_counts = {}
        for triple in self.covid_subset:
            rel = triple['relation']
            relation_counts[rel] = relation_counts.get(rel, 0) + 1

        print("\n Top 15 most common relations:")
        for rel, count in sorted(relation_counts.items(), key=lambda x: x[1], reverse=True)[:15]:
            print(f"  {rel}: {count:,}")

        # COVID-19 drug mentions
        drug_mentions = {}
        for drug in self.covid_drugs:
            count = sum(1 for triple in self.covid_subset
                       if drug.lower() in str(triple.get('head', '')).lower() or
                          drug.lower() in str(triple.get('tail', '')).lower())
            if count > 0:
                drug_mentions[drug] = count

        if drug_mentions:
            print("\n COVID-19 drug mentions:")
            for drug, count in sorted(drug_mentions.items(), key=lambda x: x[1], reverse=True):
                print(f"  {drug}: {count}")

In [ ]:
def create_covid_kg_from_prototype():
    """
    Create a COVID-19 KG based on your prototype's raw_KG structure.
    This can be used as a fallback.
    """
    raw_KG = {
        ("Nirmatrelvir and ritonavir", "treats", "COVID-19"),
        ("Remdesivir", "treats", "COVID-19"),
        ("Molnupiravir", "treats", "COVID-19"),
        ("Sotrovimab", "treats", "COVID-19"),
        ("Dexamethasone", "treats", "COVID-19"),
        ("Hydrocortisone", "treats", "COVID-19"),
        ("Prednisolone", "treats", "COVID-19"),
        ("Tocilizumab", "treats", "COVID-19"),
        ("Baricitinib", "treats", "COVID-19"),
        ("Azithromycin", "contraindicated_for", "COVID-19"),
        ("Doxycycline", "contraindicated_for", "COVID-19"),
        ("Colchicine", "contraindicated_for", "COVID-19"),
        ("Ivermectin", "contraindicated_for", "COVID-19"),
        ("Vitamin D", "contraindicated_for", "COVID-19"),
        ("Budesonide", "experimental_for", "COVID-19"),
        ("Tixagevimab plus cilgavimab", "not_recommended_for", "COVID-19"),
    }

    kg_list = []
    for h, r, t in raw_KG:
        kg_list.append({
            "head": h,
            "relation": r,
            "tail": t
        })

    # Save to data directory
    output_path = Path("./data/covid_prototype_kg.json")
    output_path.parent.mkdir(exist_ok=True)

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(kg_list, f, indent=2, ensure_ascii=False)

    print(f" Created prototype COVID-19 KG with {len(kg_list)} triples")
    print(f" Saved to: {output_path}")

    return kg_list

In [ ]:
print(" BIOS KG COVID-19 Subset Extractor")
print("=" * 50)

# Initialize with Hugging Face loading enabled
extractor = BIOSKGCOVIDSubset(load_from_hf=True)

# Load the KG
if extractor.load_kg():
    # Explore dataset structure first
    extractor.explore_dataset_structure()

    # Extract COVID-19 subset
    covid_triples = extractor.extract_covid_subset("./data/bios_kg_covid_subset.json")

    # Analyze results
    extractor.analyze_subset()

    # Print some examples
    print("\n" + "="*60)
    print("Example COVID-19 Triples")
    print("="*60)
    for i, triple in enumerate(covid_triples[:10]):
        print(f"{i+1}. Head: {triple['head']}")
        print(f"   Relation: {triple['relation']}")
        print(f"   Tail: {triple['tail']}")
        print()

else:
    print("Failed to load BIOS KG.")
    print("\n Creating COVID-19 KG from prototype as fallback...")
    covid_triples = create_covid_kg_from_prototype()